In [0]:
%run ../Copy-Datasets

In [0]:
# %run ../Copy-Datasets

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
dataset_bookstore = f"/Volumes/{current_catalog}/bookstore_eng_pro/dataset"
checkpoint_path = f"/Volumes/{current_catalog}/bookstore_eng_pro/checkpoints"
print(dataset_bookstore)

In [0]:
%sql
ALTER TABLE customers_silver 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
DESCRIBE TABLE EXTENDED customers_silver

In [0]:
%sql
DESCRIBE HISTORY customers_silver

In [0]:
import uuid

original_checkpoint_path = bookstore.checkpoint_path
bookstore.checkpoint_path = f"{original_checkpoint_path}/rerun_{uuid.uuid4().hex}"

try:
    bookstore.load_new_data()
    bookstore.process_bronze()
    bookstore.process_orders_silver()
    bookstore.process_customers_silver()
finally:
    bookstore.checkpoint_path = original_checkpoint_path

In [0]:
%sql
SELECT * 
FROM table_changes("customers_silver", 2)

In [0]:
import uuid

original_checkpoint_path = bookstore.checkpoint_path
bookstore.checkpoint_path = f"{original_checkpoint_path}/rerun_{uuid.uuid4().hex}"

try:
    bookstore.load_new_data()
    bookstore.process_bronze()
    bookstore.process_orders_silver()
    bookstore.process_customers_silver()
finally:
    bookstore.checkpoint_path = original_checkpoint_path

In [0]:
cdf_df = (spark.readStream
               .format("delta")
               .option("readChangeData", True)
               .option("startingVersion", 2)
               .table("customers_silver"))

display(cdf_df, checkpointLocation = f"{checkpoint_path}/tmp/cdf_{time.time()}")